In [6]:
import numpy as np


test1 = np.zeros((20,8,2))
print(test1.shape)
print(np.stack(test1, axis=1).shape)

(20, 8, 2)
(8, 20, 2)


In [34]:
"""Implementation"""

# Setup Imports
import pandas as pd
import numpy as np
import time
import os

import utils
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.multioutput import ClassifierChain as skl_cc
from sklearn.multioutput import MultiOutputClassifier

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from skmultilearn.problem_transform import BinaryRelevance

from sklearn.preprocessing import OneHotEncoder

from sklearn import tree
from skmultilearn.ensemble import RakelO

import scipy


from sklearn.model_selection import cross_val_predict

# Baseline Imports

from tabpfn import TabPFNClassifier

from Classifiers import ClassifierChains as cc

from Classifiers import Ensemble as en

from sklearn.metrics import jaccard_score

def main():
    files = [r"./data/PI_DataSet.txt", r"./data/INI_DataSet.txt", r"./data/NRTI_DataSet.txt",
             r"./data/NNRTI_DataSet.txt"]

    for file in files:

        # Reading in and processing high quality File
        df = pd.read_csv(file, sep='\t')

        # removing index and summary column
        df = df.iloc[:, 1:-1]

        # list of current drugs of the dataset
        drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

        # Filtering out drugs with less than 10 labels present
        unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

        if len(unusable_drugs) > 0:
            df.drop(columns=unusable_drugs, inplace=True)

            drugs = [drug for drug in drugs if drug not in unusable_drugs]

        # dropping rows with na labels
        df.dropna(subset=drugs, inplace=True)

        enc = OneHotEncoder(handle_unknown='error')



        X = df.drop(drugs, axis=1)

        enc.fit(X)
        X_trafo = enc.transform(X).toarray()
        Y = utils.get_classes(df, drugs, mode="binary")

        #clf = TabPFNClassifier()

        multi_target_pfn = cc(TabPFNClassifier, random_state=42)

        use_kfold = False

        folds = 5

        n_jobs = 4

        X_train, X_test, y_train, y_test = train_test_split(X_trafo, Y, test_size=0.33, random_state=42)

        forest = RandomForestClassifier(random_state=42)
        xgb = XGBClassifier(random_state=42)
        lr = LogisticRegression()

        models = [
            #("BR_LR", MultiOutputClassifier(lr, n_jobs=2)),
            #("BR_XGB", MultiOutputClassifier(xgb, n_jobs=2)),
            #("BR_forest", MultiOutputClassifier(forest, n_jobs=2)),
            #("CC_LR", skl_cc(lr, order="random", random_state=42, chain_method="predict_proba")),
            #("CC_xgb", skl_cc(xgb, order="random", random_state=42)),
            #("CC_forest", skl_cc(forest, order="random", random_state=42)),
            ("Rakel_lr", RakelO(base_classifier=lr, base_classifier_require_dense=[True, True], labelset_size=y_train.shape[1] // 4, model_count=6)),
            ("Rakel_xgb", RakelO(base_classifier=xgb,base_classifier_require_dense=[True, True],labelset_size=y_train.shape[1] // 4, model_count=6)),
            ("Rakel_forest", RakelO(base_classifier=forest, base_classifier_require_dense=[True, True], labelset_size=y_train.shape[1] // 4, model_count=6)),
        ]

        #ensemble = en(cc, random_state=42, n_jobs=n_jobs)

        if not use_kfold:

            for name, model in models:
                print()
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)
                print(type(y_pred))

                if isinstance(y_pred, scipy.sparse._csr.csr_matrix):
                    y_pred = y_pred.todense()

                y_pred_df = pd.DataFrame(y_pred, columns=drugs)

                y_test_df = pd.DataFrame(y_test, columns=drugs)

                #print(np.array(y_pred_proba).shape)

                utils.save_multilabel(y_pred_df, y_test_df, label=(
            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
        0] + "_" + name), path="./test/")

                if name.startswith("CC"):

                    y_pred_proba = model.predict_proba(X_test)
                    #y_pred_proba_new = np.stack(y_pred_proba, axis=1)
                    #print(y_pred_proba_new.shape)

                    y_pred_proba_new = pd.DataFrame(y_pred_proba, columns = drugs)

                    utils.save_multilabel(y_pred_proba_new, y_test_df, label=(
                        file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                    0] + "_" + name + "_probabilities"), path="./test/")
                elif name.startswith("Rakel"):
                    pass
                else:
                    y_pred_proba = model.predict_proba(X_test)

                    y_pred_proba_new = y_pred_proba

                    utils.save_multilabel_proba(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                        0] + "_" + name + "_probabilities"), path="./test/")

if __name__ == '__main__':
    main()

C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

<class 'scipy.sparse._csr.csr_matrix'>

<class 'scipy.sparse._csr.csr_matrix'>

<class 'scipy.sparse._csr.csr_matrix'>



KeyboardInterrupt: 

In [30]:
print(y_pred)

NameError: name 'y_pred' is not defined